# RAG over the MTG Comprehensive Rules: Baseline vs. Cross-Encoder Re-Ranking

corpus/dataset chosen - *Magic: The Gathering Comprehensive Rules*, Sept 2026

Two retrieval pipelines over the same chunked corpus 

- **Pipeline 1 – Baseline:** bi-encoder → top 10 by cosine similarity → top 3 to Gemini → **Answer 1**
- **Pipeline 2 – Re-ranking:** same top 10 → cross-encoder → re-sorted top 3 to Gemini → **Answer 2**

## 1. Setup

In [ ]:
import hashlib
import json
import logging
import os
import re
import time
from dataclasses import dataclass
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, display
from pypdf import PdfReader
from sentence_transformers import CrossEncoder, SentenceTransformer

/home/aditya-peketi/Documents/GitHub/lma-RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()
logging.getLogger("pypdf").setLevel(logging.ERROR)

# --- Configuration ---
PDF_PATH = Path("data/MagicCompRules.pdf")
BI_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "gemini-3-flash-preview"
LLM_CACHE = Path("cache/llm_answers.json")  # saved answers, so re runs don't spend API quota (Added this because I ran out of gemini student plan :( but later got the plus plan lol so technically not required)
REQUESTS_PER_MINUTE = 5  
MAX_CHUNK_CHARS = 1000  # rules longer than this are split on sub rule boundaries
RETRIEVE_K = 10  # candidates retrieved by the bi-encoder
CONTEXT_K = 3  # chunks passed to the LLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pd.set_option("display.max_colwidth", 120)

/home/aditya-peketi/Documents/GitHub/lma-RAG/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 2. Loading and chunking

The document is a numbered rulebook (e.g. `702.19. Trample`, with sub-rules `702.19a`, `702.19b`, …) followed by a glossary.

- **Rules:** one chunk per numbered rule. Rules longer than `MAX_CHUNK_CHARS` are split on sub-rule boundaries, producing ids like `702.19#2`. Every chunk is prefixed with its section and rule headline so split pieces keep their context.
- **Glossary:** one chunk per term. These short definitions overlap heavily with query wording, which makes them natural distractors.
- The table of contents and credits are dropped.

In [ ]:
# Regexes for rulebook's numbering scheme
RULE_RE = re.compile(r"^(\d{3}\.\d+)\.\s")
SUBRULE_RE = re.compile(r"^\d{3}\.\d+[a-z]+\s")
SECTION_RE = re.compile(r"^(\d{3})\.\s+(\S.*)$")


@dataclass
class Chunk:
    chunk_id: str
    source: str
    title: str
    text: str


def load_text(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def split_document(text):
    first = text.index("100. General")
    body_start = text.index("100. General", first + 1)
    glossary_start = text.index("\nGlossary", body_start)
    credits_start = text.index("\nCredits", glossary_start)
    return text[body_start:glossary_start], text[glossary_start + len("\nGlossary"):credits_start]

def normalize(lines):
    return re.sub(r"\s+", " ", " ".join(lines)).strip()

In [ ]:
def iter_rules(body):
    section, rule_id, lines = "", None, []
    for line in (raw.strip() for raw in body.splitlines()):
        if not line:
            continue
        rule, heading = RULE_RE.match(line), SECTION_RE.match(line)
        # A new rule or section heading starts: emit the rule collected so far
        if rule or heading:
            if rule_id:
                yield section, rule_id, lines
            rule_id, lines = (rule.group(1), [line]) if rule else (None, [])
            if heading:
                section = f"{heading.group(1)}. {heading.group(2)}"
        # Continuation line: belongs to the current rule
        elif rule_id:
            lines.append(line)
    if rule_id:
        yield section, rule_id, lines


def subrule_blocks(lines):
    blocks = [[]]
    for line in lines:
        if SUBRULE_RE.match(line) and blocks[-1]:
            blocks.append([])
        blocks[-1].append(line)
    return blocks


def pack(blocks, max_chars):
    packs = [[]]
    for block in blocks:
        if packs[-1] and len(normalize(packs[-1] + block)) > max_chars:
            packs.append([])
        packs[-1].extend(block)
    return packs


# Turn one rule into one or more chunks; extra pieces get ids like 702.19#2.
def rule_chunks(section, rule_id, lines, max_chars):
    headline = normalize(lines[:1])
    title = f"{section} > {headline}" if len(headline) <= 60 else section
    return [
        Chunk(rule_id if i == 0 else f"{rule_id}#{i + 1}", "rules", title, f"[{title}] {normalize(part)}")
        for i, part in enumerate(pack(subrule_blocks(lines), max_chars))
    ]


# One chunk per glossary term 
def glossary_chunks(glossary):
    entries = ([l.strip() for l in entry.splitlines() if l.strip()] for entry in re.split(r"\n\s*\n", glossary))
    return [
        Chunk(f"glossary:{lines[0]}", "glossary", lines[0], f"Glossary — {lines[0]}: {normalize(lines[1:])}")
        for lines in entries
        if len(lines) >= 2
    ]


# Full corpus 
def build_corpus(path, max_chars):
    body, glossary = split_document(load_text(path))
    rules = [chunk for group in iter_rules(body) for chunk in rule_chunks(*group, max_chars)]
    return rules + glossary_chunks(glossary)

In [ ]:
# Build the shared corpus and using both pipelines 
chunks = build_corpus(PDF_PATH, MAX_CHUNK_CHARS)
lengths = pd.Series([len(c.text) for c in chunks])

pd.DataFrame({
    "chunks": [len(chunks)],
    "rule chunks": [sum(c.source == "rules" for c in chunks)],
    "glossary chunks": [sum(c.source == "glossary" for c in chunks)],
    "mean chars": [int(lengths.mean())],
    "max chars": [lengths.max()],
})

,chunks,rule chunks,glossary chunks,mean chars,max chars
0,2324,1583,741,426,2914


In [ ]:
# Sanity check just for my convinience :)
pd.DataFrame([vars(c) for c in chunks if c.chunk_id.startswith("702.19")][:3] + [vars(chunks[-1])])

,chunk_id,source,title,text
0,702.19,rules,702. Keyword Abilities > 702.19. Trample,[702. Keyword Abilities > 702.19. Trample] 702.19. Trample 702.19a Trample is a static ability that modifies the rul...
1,702.19#2,rules,702. Keyword Abilities > 702.19. Trample,[702. Keyword Abilities > 702.19. Trample] 702.19b The controller of an attacking creature with trample first assign...
2,702.19#3,rules,702. Keyword Abilities > 702.19. Trample,[702. Keyword Abilities > 702.19. Trample] 702.19c Trample over planeswalkers is a variant of trample that modifies ...
3,glossary:Zone-Change Triggers,glossary,Zone-Change Triggers,Glossary — Zone-Change Triggers: Trigger events that involve objects changing zones. See rule 603.6.


## 3. Models

- **Bi-encoder** (`all-MiniLM-L6-v2`): embeds every chunk once, offline. Embeddings are L2-normalised, so inner product in a FAISS `IndexFlatIP` equals cosine similarity.
- **Cross-encoder** (`ms-marco-MiniLM-L-6-v2`): scores each (query, chunk) pair jointly at query time.
- **Generator** (Gemini): the same prompt template for both pipelines, temperature 0. Calls are spaced to stay under the free-tier rate limit; the wait is excluded from latency timings. Answers and their measured latency are cached in `cache/llm_answers.json`, so re-running the notebook does not spend API quota.

In [ ]:
# A retrieved chunk with its bi-encoder rank/score and after re-ranking cross-encoder rank/score
@dataclass
class Candidate:
    chunk: Chunk
    bi_rank: int
    bi_score: float
    ce_rank: int | None = None
    ce_score: float | None = None


# Pipeline 1 retriever: bi-encoder embeddings in a FAISS index
class DenseIndex:
    def __init__(self, model_name, chunks, device):
        self.model = SentenceTransformer(model_name, device=device)
        self.chunks = chunks
        # Embed every chunk once (also timed for reference)
        start = time.perf_counter()
        self.embeddings = self.encode([c.text for c in chunks])
        self.build_seconds = time.perf_counter() - start
        # Vectors are L2-normalised, so inner product == cosine similarity
        self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
        self.index.add(self.embeddings)

    def encode(self, texts):
        return self.model.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

    # Embed the query and return the k most similar chunks as Candidates
    def search(self, query, k):
        scores, ids = self.index.search(self.encode([query]), k)
        return [Candidate(self.chunks[i], rank, float(s)) for rank, (i, s) in enumerate(zip(ids[0], scores[0]), 1)]

In [ ]:
# Pipeline 2 stage: cross encoder that scores each (query, chunk) pair jointly
class Reranker:
    def __init__(self, model_name, device):
        self.model = CrossEncoder(model_name, device=device)

    def score(self, query, texts):
        return self.model.predict([(query, t) for t in texts], batch_size=64, show_progress_bar=False)

    # Score all candidates, sort by score (highest first) and record the new ranks
    def rerank(self, query, candidates):
        scores = self.score(query, [c.chunk.text for c in candidates])
        ranked = sorted(zip(candidates, scores), key=lambda pair: -pair[1])
        for rank, (candidate, score) in enumerate(ranked, 1):
            candidate.ce_rank, candidate.ce_score = rank, float(score)
        return [candidate for candidate, _ in ranked]

In [ ]:
# Prompt template shared by both pipelines, only {context} differs between Answer 1 and Answer 2
PROMPT = """You are a Magic: The Gathering rules judge. Answer the question using only the rules excerpts below.
Cite the rule numbers you rely on. If the excerpts do not contain enough information, say so.

Rules excerpts:
{context}

Question: {question}
Answer:"""


# Gemini wrapper 
class Generator:
    def __init__(self, model, requests_per_minute, cache_path):
        # Retry automatically on rate-limit (429) and overload (503) errors
        retry = types.HttpRetryOptions(attempts=5, initial_delay=15, max_delay=60, http_status_codes=[429, 503])
        self.client = genai.Client(api_key=os.environ["GEMINI_API_KEY"], http_options=types.HttpOptions(retry_options=retry))
        self.model = model
        # temperature=0 for repeatable answers
        self.config = types.GenerateContentConfig(temperature=0)
        # Minimum seconds between calls to stay under the free-tier limit
        self.min_interval = 60 / requests_per_minute + 1
        self.last_call = -self.min_interval
        self.cache_path = cache_path
        self.cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}

    # Sleep just long enough to respect min_interval (This is not required since now I have the plus plan)
    def wait(self):
        time.sleep(max(0.0, self.last_call + self.min_interval - time.perf_counter()))
        self.last_call = time.perf_counter()

    # One API call, only the call itself is timed
    def generate(self, prompt):
        self.wait()
        start = time.perf_counter()
        text = self.client.models.generate_content(model=self.model, contents=prompt, config=self.config).text
        return {"text": text, "seconds": time.perf_counter() - start}

    def answer(self, question, chunks):
        context = "\n\n".join(f"({i}) {c.text}" for i, c in enumerate(chunks, 1))
        prompt = PROMPT.format(context=context, question=question)
        # Cache key = hash of model + exact prompt, so any change to context or wording triggers a new call
        key = hashlib.sha256(f"{self.model}\n{prompt}".encode()).hexdigest()
        if key not in self.cache:
            self.cache[key] = self.generate(prompt)
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)
            self.cache_path.write_text(json.dumps(self.cache, indent=1))
        return self.cache[key]["text"], self.cache[key]["seconds"]

In [10]:
# Load models and embed the corpus
index = DenseIndex(BI_ENCODER, chunks, DEVICE)
reranker = Reranker(CROSS_ENCODER, DEVICE)
generator = Generator(LLM_MODEL, REQUESTS_PER_MINUTE, LLM_CACHE)

print(f"Indexed {index.index.ntotal} chunks in {index.build_seconds:.2f}s on {DEVICE}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3733.77it/s]


Indexed 2324 chunks in 28.60s on cpu


## 4. Pipelines

`run_query` runs both pipelines on one query and times every stage. Both pipelines share the initial top-10 retrieval, so Pipeline 2 only adds the re-ranking step.

In [ ]:
# Everything produced for one query: basically both rankings, stage timings and both answers
@dataclass
class QueryResult:
    query: str
    baseline: list
    reranked: list
    timings: dict
    answers: dict


# return the time
def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


# Run both pipelines on one query. They share the same top-10 retrieval.
# Pipeline 2 only adds the re-ranking step.
def run_query(query, generate=True):
    candidates, t_retrieve = timed(index.search, query, RETRIEVE_K)
    reranked, t_rerank = timed(reranker.rerank, query, candidates)
    timings = {"retrieve": t_retrieve, "rerank": t_rerank}
    answers = {}
    if generate:
        # Answer 1: top 3 in bi-encoder order. Answer 2: top 3 in cross-encoder order
        answers["baseline"], timings["llm_baseline"] = generator.answer(query, [c.chunk for c in candidates[:CONTEXT_K]])
        answers["reranked"], timings["llm_reranked"] = generator.answer(query, [c.chunk for c in reranked[:CONTEXT_K]])
    return QueryResult(query, candidates, reranked, timings, answers)

In [ ]:
def contains_rule(candidate, rule):
    return f"{rule} " in candidate.chunk.text

def target_ranks(result, rule):
    hit = next((c for c in result.baseline if contains_rule(c, rule)), None)
    return (hit.bi_rank, hit.ce_rank, hit.chunk.chunk_id) if hit else (None, None, None)

def rank_table(result, rule):
    return pd.DataFrame([
        {
            "chunk": c.chunk.chunk_id,
            "bi rank": c.bi_rank,
            "cosine": round(c.bi_score, 3),
            "ce rank": c.ce_rank,
            "ce score": round(c.ce_score, 2),
            "target": "◀" if contains_rule(c, rule) else "",
            "text": c.chunk.text[:110],
        }
        for c in result.baseline
    ]).set_index("chunk")

def show_answers(result):
    for name, top in [("baseline", result.baseline), ("reranked", result.reranked)]:
        context = ", ".join(c.chunk.chunk_id for c in top[:CONTEXT_K])
        display(Markdown(f"**{name.title()} answer** — context: `{context}`\n\n{result.answers[name]}"))

## 5. Failure-mode stress tests

Each query has a **target rule**: the sub-rule that actually answers it. A query counts as a failure case when the target sits at rank 4–10 after bi-encoder retrieval (so the baseline LLM never sees it) and the cross-encoder promotes it into the top 3.

| Category | Why the bi-encoder struggles here |
|---|---|
| Keyword trap | The query shares words with other passages; the answer uses different terminology |
| Exception | The general rule outranks the specific "except / unless / normally" clause |
| Negation | "required", "can't", "not enough" are weak signals in a pooled embedding |
| Buried chunk | Short glossary entries and near-duplicates crowd the true rule out of the top 3 |

In [ ]:
# Failure queries that I have tested
FAILURE_QUERIES = [
    ("Keyword trap", "Can I respond to my opponent playing a land?", "116.2a"),
    ("Keyword trap", "Does an attacking creature stay attacking if it stops being a creature?", "506.4."),
    ("Exception", "Does deathtouch kill a creature with indestructible?", "702.12b"),
    ("Exception", "Is there ever a situation where players get priority during cleanup?", "514.3a"),
    ("Negation", "Is the defending player required to block?", "509.1a"),
    ("Negation", "Can I pay part of a cost if I don't have enough mana?", "118.3."),
    ("Buried chunk", "Can a player concede at any time?", "104.3a"),
    ("Buried chunk", "Do I lose immediately when my library is empty?", "121.4."),
]

# Run both pipelines including LLM answers for every failure query
results = {query: run_query(query) for _, query, _ in FAILURE_QUERIES}

In [ ]:
def shift_summary(queries, results):
    rows = []
    for category, query, rule in queries:
        before, after, chunk_id = target_ranks(results[query], rule)
        rows.append({"category": category, "query": query, "target rule": rule, "chunk": chunk_id,
                     "baseline rank": before, "re-ranked rank": after,
                     "shift": f"#{before} → #{after}" if before else "not retrieved"})
    return pd.DataFrame(rows).astype({"baseline rank": "Int64", "re-ranked rank": "Int64"})


shift_summary(FAILURE_QUERIES, results)

,category,query,target rule,chunk,baseline rank,re-ranked rank,shift
0,Keyword trap,Can I respond to my opponent playing a land?,116.2a,116.2,9,3,#9 → #3
1,Keyword trap,Does an attacking creature stay attacking if it stops being a creature?,506.4.,506.4,7,3,#7 → #3
2,Exception,Does deathtouch kill a creature with indestructible?,702.12b,702.12,5,3,#5 → #3
3,Exception,Is there ever a situation where players get priority during cleanup?,514.3a,514.3,5,1,#5 → #1
4,Negation,Is the defending player required to block?,509.1a,509.1,9,1,#9 → #1
5,Negation,Can I pay part of a cost if I don't have enough mana?,118.3.,118.3,7,1,#7 → #1
6,Buried chunk,Can a player concede at any time?,104.3a,104.3,8,3,#8 → #3
7,Buried chunk,Do I lose immediately when my library is empty?,121.4.,121.4,5,2,#5 → #2


### Per-query detail

Full top-10 with both scores, followed by Answer 1 (baseline) and Answer 2 (re-ranked).

In [ ]:
for category, query, rule in FAILURE_QUERIES:
    display(Markdown(f"#### {category}: *{query}*"))
    display(rank_table(results[query], rule))
    show_answers(results[query])

#### Keyword trap: *Can I respond to my opponent playing a land?*

,bi rank,cosine,ce rank,ce score,target,text
chunk,,,,,,
305.3,1,0.662,2,2.30,,"[305. Lands] 305.3. A player can’t play a land, for any reason, if it isn’t their turn. Ignore any part of an"
305.4,2,0.597,7,-1.56,,[305. Lands] 305.4. Effects may also allow players to “put” lands onto the battlefield. This isn’t the same as
305.1,3,0.597,1,4.64,,[305. Lands] 305.1. A player who has priority may play a land card from their hand during a main phase of thei
305.2,4,0.553,6,-1.31,,"[305. Lands] 305.2. A player can normally play one land during their turn; however, continuous effects may inc"
305.9,5,0.526,9,-2.42,,"[305. Lands] 305.9. If an object is both a land and another card type, it can be played only as a land. It can"
701.18,6,0.525,4,-0.31,,[701. Keyword Actions > 701.18. Play] 701.18. Play 701.18a To play a land means to put it onto the battlefield
glossary:Play,7,0.480,8,-2.34,,"Glossary — Play: 1. To play a land is to put a land onto the battlefield as a special action. See rule 116, “S"
505.6,8,0.474,5,-0.35,,"[505. Main Phase] 505.6. Fourth, the active player gets priority. (See rule 117, “Timing and Priority.”) 505.6"
116.2,9,0.432,3,-0.22,◀,[116. Special Actions > 116.2. There are twelve special actions:] 116.2. There are twelve special actions: 116


**Baseline answer** — context: `305.3, 305.4, 305.1`

No, you cannot respond to your opponent playing a land.

According to rule **305.1**, playing a land is a special action that does not use the stack. Because the land does not go on the stack, it is not a spell, and "players can’t respond to it with instants or activated abilities."

**Reranked answer** — context: `305.1, 305.3, 116.2`

No, you cannot respond to your opponent playing a land.

According to rule **305.1**, playing a land is a special action that does not use the stack. Because it does not go on the stack, it is not a spell, and "players can’t respond to it with instants or activated abilities." Rule **116.2a** also confirms that playing a land is a special action.

#### Keyword trap: *Does an attacking creature stay attacking if it stops being a creature?*

...

**Reranked answer** — context: `723.6, 101.1, 104.3`

Yes, a player can concede the game at any time.

According to the rules provided:
*   **104.3a** states: "A player can concede the game at any time."
*   **723.6** confirms this, noting that "A player may concede the game at any time, even if they are controlled by another player."
*   **101.1** establishes that this is a fundamental rule that cards cannot override, stating: "The only exception [to card text taking precedence over rules] is that a player can concede the game at any time."

#### Buried chunk: *Do I lose immediately when my library is empty?*

,bi rank,cosine,ce rank,ce score,target,text
chunk,,,,,,
401.6,1,0.405,3,-8.20,,"[401. Library] 401.6. If an effect causes a player to play with the top card of their library revealed, and th"
701.24,2,0.393,7,-9.42,,[701. Keyword Actions > 701.24. Shuffle] 701.24. Shuffle 701.24a To shuffle a library or a face-down pile of c
401.1,3,0.383,9,-10.35,,"[401. Library] 401.1. When a game begins, each player’s deck becomes their library."
401.2,4,0.358,6,-9.10,,[401. Library] 401.2. Each library must be kept in a single face-down pile. Players can’t look at or change th
121.4,5,0.352,2,-8.08,◀,[121. Drawing a Card] 121.4. A player who attempts to draw a card from a library with no cards in it loses the
701.24#3,6,0.349,4,-8.82,,[701. Keyword Actions > 701.24. Shuffle] 701.24d If an effect would cause a player to shuffle a set of objects
104.5,7,0.332,10,-11.17,,"[704. See also rule 903.10.)] 104.5. If a player loses the game, that player leaves the game. If the game is a"
401.3,8,0.330,5,-8.89,,[401. Library] 401.3. Any player may count the number of cards remaining in any player’s library at any time.
727.3,9,0.324,1,-4.39,,"[727. Restarting the Game] 727.3. Because each player draws seven cards when the new game begins, any player w"


**Baseline answer** — context: `401.6, 701.24, 401.1`

The provided excerpts do not contain enough information to answer this question. None of the rules provided (401.1, 401.6, or 701.24) describe the conditions under which a player loses the game.

**Reranked answer** — context: `727.3, 121.4, 401.6`

No, you do not lose immediately simply because your library is empty.

According to rule **121.4**, a player loses the game only when they **attempt to draw a card** from a library with no cards in it. This loss occurs "the next time a player would receive priority" as a state-based action.

Additionally, rule **727.3** notes that if a game has been restarted and a player has fewer than seven cards in their library, they will lose when state-based actions are checked during the upkeep step of the first turn (due to the requirement to draw seven cards at the start of the game).

## 6. Where re-ranking does not help

Two control cases, for balance:

- **Re-ranker regression:** the bi-encoder already has the target at #1 and the cross-encoder pushes it down.
- **Recall miss:** the target never reaches the bi-encoder's top 10, so re-ranking cannot recover it. Stage-two quality is capped by stage-one recall.

In [ ]:
# Control cases where re ranking does not help
CONTROL_QUERIES = [
    ("Re-ranker regression", "Can a counterspell counter an ability?", "701.6a"),
    ("Recall miss", "Is a creature with 0 toughness destroyed?", "704.5f"),
]

control_results = {query: run_query(query) for _, query, _ in CONTROL_QUERIES}
shift_summary(CONTROL_QUERIES, control_results)

,category,query,target rule,chunk,baseline rank,re-ranked rank,shift
0,Re-ranker regression,Can a counterspell counter an ability?,701.6a,701.6,1,4,#1 → #4
1,Recall miss,Is a creature with 0 toughness destroyed?,704.5f,NaN,<NA>,<NA>,not retrieved


In [ ]:
# Rank of the target among ALL chunks and not just the top 10, this is to show how far off a recall miss is
def full_rank(query, rule):
    order = np.argsort(-(index.embeddings @ index.encode([query])[0]))
    return next(rank for rank, i in enumerate(order, 1) if f"{rule} " in chunks[i].text)


for category, query, rule in CONTROL_QUERIES:
    print(f"{category}: target {rule} sits at bi-encoder rank #{full_rank(query, rule)} out of {len(chunks)}")

Re-ranker regression: target 701.6a sits at bi-encoder rank #1 out of 2324
Recall miss: target 704.5f sits at bi-encoder rank #82 out of 2324


In [18]:
# Same per-query detail for the control cases
for _, query, rule in CONTROL_QUERIES:
    display(Markdown(f"#### *{query}*"))
    display(rank_table(control_results[query], rule))
    show_answers(control_results[query])

#### *Can a counterspell counter an ability?*

,bi rank,cosine,ce rank,ce score,target,text
chunk,,,,,,
701.6,1,0.735,4,4.06,◀,[701. Keyword Actions > 701.6. Counter] 701.6. Counter 701.6a To counter a spell or ability means to cancel it
glossary:Counter,2,0.733,7,2.60,,Glossary — Counter: 1. To cancel a spell or ability so it doesn’t resolve and none of its effects occur. See r
113.9,3,0.627,3,4.33,,"[113. Abilities] 113.9. Activated and triggered abilities on the stack aren’t spells, and therefore can’t be c"
122.8,4,0.558,2,5.25,,[122. Counters] 122.8. If a triggered ability instructs a player to put one object’s counters on another objec
122.1,5,0.530,5,3.85,,[122. Counters] 122.1. A counter is a marker placed on an object or player that modifies its characteristics a
122.6,6,0.528,6,3.71,,[122. Counters] 122.6. Some spells and abilities refer to counters being put on an object. This refers to putt
glossary:Stun Counter,7,0.512,10,-4.67,,Glossary — Stun Counter: A counter that stops a permanent from untapping. See rule 122.1d.
113.11,8,0.502,8,0.53,,[113. Abilities] 113.11. Effects can stop an object from having a specified ability. These effects say that th
122.9,9,0.492,1,5.31,,[122. Counters] 122.9. If an activated ability of an object instructs a player to put its counters on another


**Baseline answer** — context: `701.6, glossary:Counter, 113.9`

Yes, an ability can be countered, but it depends on the type of ability and the specific wording of the counter effect:

*   **Activated and triggered abilities** on the stack can be countered by effects that specifically counter abilities (113.9).
*   **Activated and triggered abilities** cannot be countered by effects that counter only spells (113.9).
*   **Static abilities** do not use the stack and cannot be countered at all (113.9).

When an ability is countered, it is removed from the stack, it does not resolve, and none of its effects occur (701.6a, Glossary). The player who activated the ability does not receive a refund of any costs paid (701.6b).

**Reranked answer** — context: `122.9, 122.8, 113.9`

Based on the rules provided:

Activated and triggered abilities cannot be countered by anything that counters only spells, because these abilities are not spells. However, they can be countered by effects that specifically counter abilities. Static abilities cannot be countered at all because they do not use the stack.

**Rule cited:**
*   **113.9**: "Activated and triggered abilities on the stack aren’t spells, and therefore can’t be countered by anything that counters only spells. Activated and triggered abilities on the stack can be countered by effects that specifically counter abilities. Static abilities don’t use the stack and thus can’t be countered at all."

#### *Is a creature with 0 toughness destroyed?*

,bi rank,cosine,ce rank,ce score,target,text
chunk,,,,,,
glossary:Lethal Damage,1,0.651,8,2.16,,Glossary — Lethal Damage: An amount of damage greater than or equal to a creature’s toughness. See rules 120.4
302.4,2,0.622,4,3.90,,[302. Creatures] 302.4. Power and toughness are characteristics only creatures have. 302.4a A creature’s power
208.5,3,0.613,2,4.79,,"[208. Power/Toughness] 208.5. If a creature somehow has no value for its power, its power is 0. The same is tr"
208.2#3,4,0.569,7,2.20,,[208. Power/Toughness] 208.2b The card may have a static ability that creates a replacement effect that sets t
glossary:Toughness,5,0.568,10,-0.32,,Glossary — Toughness: 1. Part of a card that only creature cards have. A creature card’s toughness is printed
302.7,6,0.541,1,5.24,,[302. Creatures] 302.7. Damage dealt to a creature by a source with neither wither nor infect is marked on tha
208.1,7,0.537,5,3.37,,[208. Power/Toughness] 208.1. A creature card has two numbers separated by a slash printed in its lower right
120.6,8,0.519,3,4.43,,"[120. Damage] 120.6. Damage marked on a creature remains until the cleanup step, even if that permanent stops"
701.10,9,0.500,6,2.24,,[701. Keyword Actions > 701.10. Double] 701.10. Double 701.10a Doubling a creature’s power and/or toughness cr


**Baseline answer** — context: `glossary:Lethal Damage, 302.4, 208.5`

The provided excerpts do not contain enough information to determine if a creature with 0 toughness is destroyed.

While the excerpts provide the following definitions, they do not state the outcome for a creature that has 0 toughness:
*   **Rule 302.4b** states that "A creature’s toughness is the amount of damage needed to destroy it."
*   **Rule 208.5** states that a creature's toughness can be 0.
*   **The Glossary (1)** defines "Lethal Damage" as damage "greater than or equal to a creature’s toughness."

The excerpts do not explain what happens to a creature with 0 toughness in the absence of damage, nor do they provide the text for the rules governing destruction or state-based actions (such as rule **704.5g**, which is referenced in the Glossary but not included in the excerpts).

**Reranked answer** — context: `302.7, 208.5, 120.6`

The provided excerpts do not contain enough information to determine if a creature with 0 toughness is destroyed.

According to the rules provided:
*   **Rule 208.5** states that if a creature has no value for its toughness, its toughness is 0.
*   **Rules 302.7 and 120.6** describe the conditions for "lethal damage," stating that if the total damage marked on a creature is greater than or equal to its toughness, it is destroyed as a state-based action.

However, the excerpts are insufficient for the following reasons:
1.  **Damage Requirement:** Rules 302.7 and 120.6 only define destruction in the context of damage being marked on a creature. Rule 302.7 specifies that "Damage dealt to a creature... is marked on that creature." The excerpts do not define the "total damage marked" for a creature that has not been dealt damage, nor do they state that a creature with 0 toughness is destroyed in the absence of damage.
2.  **Missing State-Based Action:** In the full Magic: The Gathering rules, a creature with 0 toughness is put into the graveyard by a specific state-based action (Rule 704.5f) that is distinct from destruction due to lethal damage (Rule 704.5g). Because the provided excerpts only cover the lethal damage rule and not the 0-toughness rule, they do not contain the information necessary to determine the fate of a creature that simply has 0 toughness.

## 7. Latency profiling

Retrieval and re-ranking are timed over `REPEATS` runs of every query after a warm-up pass. LLM latency comes from the single generation run above (network-bound, so it is reported separately).

In [ ]:
# Time retrieval and re ranking over many runs for stable numbers for measurements lol
REPEATS = 20
all_queries = [q for _, q, _ in FAILURE_QUERIES + CONTROL_QUERIES]

for query in all_queries:
    run_query(query, generate=False)

# Collect stage timings in milliseconds 
samples = pd.DataFrame([run_query(q, generate=False).timings for q in all_queries for _ in range(REPEATS)]) * 1000
# LLM latency recorded when each answer was first generated (stored in the cache)
llm = pd.DataFrame([r.timings for r in {**results, **control_results}.values()])[["llm_baseline", "llm_reranked"]] * 1000

# Pipeline 1 = retrieval only; Pipeline 2 = retrieval + re-ranking
stages = pd.DataFrame({
    "Pipeline 1: bi-encoder top 10": samples["retrieve"],
    "Cross-encoder re-rank (10 pairs)": samples["rerank"],
    "Pipeline 2: bi-encoder + re-rank": samples["retrieve"] + samples["rerank"],
})
latency = stages.describe(percentiles=[0.5, 0.95]).T[["mean", "50%", "95%"]].round(2)
latency.columns = ["mean ms", "median ms", "p95 ms"]
latency

,mean ms,median ms,p95 ms
Pipeline 1: bi-encoder top 10,9.40,9.44,12.21
Cross-encoder re-rank (10 pairs),281.64,285.16,350.33
Pipeline 2: bi-encoder + re-rank,291.04,294.81,359.87


In [ ]:
end_to_end = pd.DataFrame({
    "Pipeline 1 (retrieve + LLM)": samples["retrieve"].mean() + llm["llm_baseline"],
    "Pipeline 2 (retrieve + re-rank + LLM)": samples["retrieve"].mean() + samples["rerank"].mean() + llm["llm_reranked"],
})
end_to_end.describe().T[["mean", "50%", "max"]].round(0).rename(columns=lambda c: f"{c} ms")

,mean ms,50% ms,max ms
Pipeline 1 (retrieve + LLM),14987.0,10866.0,32898.0
Pipeline 2 (retrieve + re-rank + LLM),21385.0,12432.0,66309.0


## 8. Scaling: cross-encoding the whole corpus

The bi-encoder embeds the corpus once offline; each query costs one encode plus a vector search. A cross-encoder has nothing to pre-compute: every query needs a forward pass for *every* (query, chunk) pair. Below, the cross-encoder is run over all chunks for one query and compared with re-ranking 10 candidates.

In [ ]:
query = FAILURE_QUERIES[0][1]
texts = [c.text for c in chunks]

reranker.score(query, texts[:64])
_, t_full = timed(reranker.score, query, texts)
_, t_top10 = timed(reranker.score, query, texts[:RETRIEVE_K])
_, t_bi = timed(index.search, query, RETRIEVE_K)

pd.DataFrame({
    "items scored": [len(texts), RETRIEVE_K, len(texts)],
    "seconds per query": [t_full, t_top10, t_bi],
}, index=["Cross-encoder over full corpus", "Cross-encoder over top 10", "Bi-encoder search over full corpus"]).round(4)

,items scored,seconds per query
Cross-encoder over full corpus,2324,40.7734
Cross-encoder over top 10,10,0.2308
Bi-encoder search over full corpus,2324,0.0063


In [ ]:
# Extrapolate cross-encoder cost per query to larger corpora linear in size. Mentioned in the report
per_pair = t_full / len(texts)
pd.DataFrame({
    "corpus size": [10**3, 10**5, 10**7],
    "est. cross-encoder seconds per query": [n * per_pair for n in (10**3, 10**5, 10**7)],
}).round(2)

,corpus size,est. cross-encoder seconds per query
0,1000,17.54
1,100000,1754.45
2,10000000,175445.13
